# Xuất lại checkpoint tầng 2 và tầng 3 để tải về được

Notebook này **không huấn luyện và không cần GPU**. Nó chỉ đọc checkpoint từ output
của hai kernel cũ (khai trong `kernel_sources`), kiểm tra trọng số còn nguyên vẹn,
rồi chép sang `/kaggle/working` để tạo Dataset.

## Vì sao cần

`kaggle kernels output` tải về **file trọng số 0 byte**: mọi file nhỏ (config,
`head.pt`, `bpe.codes`, `dict.txt`) đều thật, riêng `model.safetensors` rỗng. Với
BARTpho, CLI còn bỏ sót hẳn thư mục `final/` mà các kernel khác vẫn đang dùng.

Nhưng `tang4` (14/09) và `tang2_score` (15/09) đọc được checkpoint qua
`kernel_sources` và chạy thành công, nên trọng số **vẫn còn bên phía Kaggle**.
Notebook này xác nhận điều đó, và nếu đúng thì **không phải huấn luyện lại**.

## Settings

| Mục | Đặt thành |
|---|---|
| **Accelerator** | `None` — chỉ chép file, dùng GPU là phí quota |
| **Internet** | `Off` |
| **Add data** | output của `dl-summarisevn-vit5` và `dl-summarisevn-tang2` |

In [ ]:
# ==== CHI SUA O NAY ===================================================
NGUON = {
    "bartpho": "/kaggle/input/**/vinai_bartpho-syllable_train_20k/final",
    "tang2": "/kaggle/input/**/phobert_sent/final",
}
RA = "/kaggle/working/ckpt"
# ======================================================================

# Ten file trong so; phai co it nhat mot cai KHAC 0 byte thi checkpoint moi dung duoc.
TRONG_SO = ("model.safetensors", "pytorch_model.bin")

print("Se xuat:", list(NGUON))

In [ ]:
import glob
import os

TIM = {}
for ten, mau in NGUON.items():
    found = sorted(glob.glob(mau, recursive=True))
    if not found:
        raise RuntimeError(
            f"Khong thay checkpoint khop {mau}.\n"
            "Vao Add-ons > Add data > Your Work, them output cua ca hai kernel\n"
            "dl-summarisevn-vit5 va dl-summarisevn-tang2.\n"
            f"Hien /kaggle/input co: {sorted(glob.glob('/kaggle/input/*/*'))[:20]}")
    TIM[ten] = found[0]
    print(f"{ten:8s} -> {found[0]}")

In [ ]:
# Chot chan: trong so phai KHAC 0 byte. Day la cau hoi ma notebook nay sinh ra de tra
# loi -- neu o day cung 0 byte thi trong so da mat that va buoc phai huan luyen lai.
for ten, duong in TIM.items():
    print(f"\n=== {ten}: {duong}")
    tot = False
    for f in sorted(os.listdir(duong)):
        p = os.path.join(duong, f)
        if os.path.isfile(p):
            mb = os.path.getsize(p) / 1e6
            print(f"   {f:32s} {mb:9.2f} MB")
            if f in TRONG_SO and os.path.getsize(p) > 0:
                tot = True
    if not tot:
        raise RuntimeError(
            f"{ten}: KHONG co file trong so nao khac 0 byte trong {duong}.\n"
            "Trong so da mat that -> phai chay lai kernel huan luyen.")
    print(f"   => {ten}: trong so CON NGUYEN VEN")

In [ ]:
import shutil

os.makedirs(RA, exist_ok=True)
for ten, duong in TIM.items():
    dich = os.path.join(RA, ten)
    if os.path.isdir(dich):
        shutil.rmtree(dich)
    shutil.copytree(duong, dich)
    n = sum(os.path.getsize(os.path.join(r, f))
            for r, _, fs in os.walk(dich) for f in fs)
    print(f"{ten:8s} -> {dich}  ({n / 1e6:.1f} MB)")

tong = sum(os.path.getsize(os.path.join(r, f))
           for r, _, fs in os.walk(RA) for f in fs)
print(f"\nTONG: {tong / 1e6:.1f} MB trong {RA}")
if tong < 100e6:
    raise RuntimeError("Tong duoi 100 MB -- gan nhu chac chan trong so khong duoc chep.")

## Sau khi chạy

Nếu ô kiểm báo **trọng số còn nguyên vẹn**, vào mục *Output* của kernel này, chọn thư
mục `ckpt` rồi **Create Dataset** từ nó. Dataset phục vụ file lớn đáng tin cậy, nên
sau đó tải về máy bằng:

    kaggle datasets download -d minh12605/<ten-dataset> -p ~/.cache/dl-summarisevn --unzip

Nếu ô kiểm **báo lỗi 0 byte** thì trọng số đã mất thật, và cách duy nhất là chạy lại
kernel huấn luyện (`dl-summarisevn-vit5` cho BARTpho, `dl-summarisevn-tang2` cho tầng 2).